# Rootbeer — Google trends CSVs (`google*.csv`)
Matches the Designer input **google\*.csv**: unions all `google_rootbeer_YYYY.csv` files with the same schema.

**Prerequisite:** **`notebook_00_setup_configuration`** is `%run` below. The union result is saved to **`bronze_table_google_trends`** (see `pipeline_config`).

In [0]:
%run ./notebook_00_setup_configuration

# 00 — Setup and configuration (Alteryx → Databricks)

**Purpose:** Unity Catalog names, ADLS-backed storage paths, Spark settings, and a shared `pipeline_config` temp view for downstream notebooks (`01` bronze, `02` silver, etc.).

**Naming convention**
- **Catalog** — top-level UC boundary (e.g. training workspace catalog).
- **Schemas** — `{user}_{layer}_alteryx_rootbeer`: bronze = raw Designer exports, silver = cleansed joins / macro parity, gold = KPIs and reporting tables.
- **Tables** — `{layer}_alteryx_rootbeer_{entity}` (e.g. `bronze_alteryx_rootbeer_transaction`). **Fact** and **dimension** tables from config are written in **`SILVER_SCHEMA`** in notebook 01 (star-schema naming); **gold** tables hold aggregates.
- **Paths** — under an ADLS container exposed as a **Unity Catalog external volume** or **`abfss://`** URI. Adjust `STORAGE_ACCOUNT` and `ADLS_CONTAINER` to match your landing zone.

## Configuration parameters

✅ Configuration loaded
   Catalog: `na-dbxtraining`
   Bronze schema: biju_bronze
   Silver schema: biju_silver
   Gold schema: biju_gold
   Source (Volume): /Volumes/na-dbxtraining/biju_raw/biju_vol/alteryx/rootbeer/source/
   ADLS base (abfss): abfss://landingzone-biju@dlsdbxtraining002.dfs.core.windows.net/alteryx


## Create schemas

✅ Schemas ensured:
   Bronze: ``na-dbxtraining``.biju_bronze
   Silver: ``na-dbxtraining``.biju_silver
   Gold:   ``na-dbxtraining``.biju_gold


## Spark configuration

## Save configuration for other notebooks

✅ Configuration saved to temp view `pipeline_config`

📋 Full configuration:
   adls_base_path: abfss://landingzone-biju@dlsdbxtraining002.dfs.core.windows.net/alteryx
   adls_container: landingzone-biju
   broadcast_threshold: 10485760
   bronze_checkpoint_path: /Volumes/na-dbxtraining/biju_raw/biju_vol/alteryx/rootbeer/checkpoints/bronze/
   bronze_schema: biju_bronze
   bronze_table_customer: bronze_alteryx_rootbeer_customer
   bronze_table_geolocation: bronze_alteryx_rootbeer_geolocation
   bronze_table_google_trends: bronze_alteryx_rootbeer_google_trends
   bronze_table_rootbeer: bronze_alteryx_rootbeer_product
   bronze_table_rootbeer_review: bronze_alteryx_rootbeer_review
   bronze_table_transaction: bronze_alteryx_rootbeer_transaction
   catalog: `na-dbxtraining`
   checkpoint_location: /Volumes/na-dbxtraining/biju_raw/biju_vol/alteryx/rootbeer/checkpoints/bronze/
   cluster_columns_csv: transaction_date,creditcard_type,location_id
   dim_creditcard_type: dim_alteryx_rootbeer_cr

## Ready

- **`notebook_01_rootbeer_transactions_pipeline.py`** and **`notebook_02_google_trends_union.py`** call `%run ./notebook_00_setup_configuration` and read paths from **`pipeline_config`** (`source_data_path`, etc.). You can still run this notebook alone to refresh schemas, Spark settings, and the temp view.
- **ADLS:** External location / volume should cover `abfss://alteryx@dlsdbxtraining002.dfs.core.windows.net/` (container `alteryx`, account `dlsdbxtraining002`). Subfolder `rootbeer/` matches `ADLS_BASE_PATH`.

In [0]:
%run ./transform

# transform — Generic DataFrame Transformation Utilities
Shared helper library for all Alteryx-converted notebooks.
Use via `%run ./transform` at the top of any notebook.

[transform] Utilities loaded: read_csv, write_output, save_as_delta_table,
            save_as_uc_table_from_config, filter_rows, drop_nulls, drop_duplicates,
            join_dfs, add_column, rename_cols, cast_cols, select_cols, summarize, preview
[transform] Rootbeer macros: state_names_dataframe, macro_imputation_v3_fill_null,
            macro_multi_field_formula_brand_urls, macro_cleanse_review_text,
            macro_batch_summarize_revenue_profit, batch_macro_creditcard_revenue_profit_from_volume,
            add_placeholder_profit, standard_macro_maven_join,
            build_rootbeer_enriched_transactions, read_google_rootbeer_union


In [0]:
_rows = spark.sql("SELECT * FROM pipeline_config LIMIT 1").collect()
if not _rows:
    raise RuntimeError("pipeline_config is empty; run notebook_00_setup_configuration first.")
cfg = _rows[0].asDict()
BASE_PATH = cfg["source_data_path"].rstrip("/")
print(f"Using source_data_path: {BASE_PATH}")

Using source_data_path: /Volumes/na-dbxtraining/biju_raw/biju_vol/alteryx/rootbeer/source


In [0]:
google_trends = read_google_rootbeer_union(BASE_PATH)
preview(google_trends, 20, label="google trends (union)")

[google trends (union)] Schema:
root
 |-- Category: All categories: string (nullable = true)

[google trends (union)] Sample (20 rows):
+------------------------+
|Category: All categories|
+------------------------+
|Week                    |
|2018-01-07              |
|2018-01-14              |
|2018-01-21              |
|2018-01-28              |
|2018-02-04              |
|2018-02-11              |
|2018-02-18              |
|2018-02-25              |
|2018-03-04              |
|2018-03-11              |
|2018-03-18              |
|2018-03-25              |
|2018-04-01              |
|2018-04-08              |
|2018-04-15              |
|2018-04-22              |
|2018-04-29              |
|2018-05-06              |
|2018-05-13              |
+------------------------+
only showing top 20 rows
[google trends (union)] Total rows: 265


DataFrame[Category: All categories: string]

In [0]:
import re

# Persist to bronze (config): single Delta table for all years unioned
cfg["catalog"] = cfg["catalog"].strip("`")

_gt_clean = google_trends
for c in _gt_clean.columns:
    _gt_clean = _gt_clean.withColumnRenamed(c, re.sub(r'[^a-zA-Z0-9_]', '_', c))

save_as_uc_table_from_config(
    _gt_clean,
    cfg,
    schema_key="bronze_schema",
    table_key="bronze_table_google_trends",
)

# Optional: silver copy for a cleansed layer name (same data; use if you add transforms later)
save_as_uc_table_from_config(
    _gt_clean,
    cfg,
    schema_key="silver_schema",
    table_key="silver_table_google_trends",
)

[save_as_uc_table_from_config] `na-dbxtraining`.`biju_bronze`.`bronze_alteryx_rootbeer_google_trends`
[save_as_uc_table_from_config] `na-dbxtraining`.`biju_silver`.`silver_alteryx_rootbeer_google_trends_union`
